In [1]:
# Ячейка 1: Импорты и константы

import gspread
import pandas as pd
from datetime import datetime, date
from google.oauth2.service_account import Credentials
from pathlib import Path
import re

# Константы подключения (синхронизированы с Парсинг_v4)
SPREADSHEET_ID = "1wxNIhrPjoFQf3vqatu9oK78W5RmNoI3YG6sf9i4GWAQ"
SHEET_NAMES = [
    "DEMO Лист", "A-Benefits", "1C", "Active Bank", "A-Sber",
    "Betta Bank", "Evergreen Bank+Laba ABS", "Liberty Bank",
    "Investment", "L&D", "MyCare", "SQL Trainee", "ProBusiness",
    "Уволенные 2026", "Стаффинг",
]
INTERIM_STATUS_SHEET = "Промежуточный статус стажера"

# Индексы столбцов основных листов (начинаются с 0)
COL_TECH = 1      # B
COL_TECHLEAD = 2  # C
COL_NAME = 3      # D
COL_START = 5     # F
COL_STOP = 6      # G
COL_REASON = 7    # H

# Индексы столбцов листа "Промежуточный статус стажера"
INTERIM_COL_TECH = 2         # C
INTERIM_COL_TECHLEAD = 3     # D
INTERIM_COL_NAME = 4         # E
INTERIM_COL_START = 6        # G
INTERIM_COL_STOP = 7         # H
INTERIM_COL_REASON = 8       # I

# Хранилище снепшотов
SNAPSHOT_DIR = Path("snapshots")
SNAPSHOT_DIR.mkdir(exist_ok=True)
HISTORY_FILE = SNAPSHOT_DIR / "history.csv"
CHANGES_FILE = SNAPSHOT_DIR / "changes_log.csv"

# Состав колонок снепшота
SNAPSHOT_COLUMNS = [
    "Дата_снепшота", "Вид_записи", "Источник",
    "Технология_норм", "Технология_оригинал",
    "Техлид_норм", "Техлид_оригинал",
    "Лаборант_норм", "Лаборант_оригинал",
    "Дата_старта", "Дата_стопа", "Причина_статус",
]
KEY_COLS = ["Вид_записи", "Источник", "Технология_норм", "Техлид_норм", "Лаборант_норм"]
COMPARE_COLS = ["Дата_старта", "Дата_стопа", "Причина_статус"]
CHANGES_COLUMNS = ["Дата_снепшота", "Тип_изменения"] + KEY_COLS + ["Было", "Стало"]

In [2]:
# Ячейка 2: Функции парсинга и очистки (синхронизированы с Парсинг_v4)

def authenticate_with_service_account(credentials_file: str) -> gspread.Client:
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly"
    ]
    creds = Credentials.from_service_account_file(credentials_file, scopes=scopes)
    return gspread.authorize(creds)

def clean_date(val) -> date | None:
    if pd.isna(val) or str(val).strip() == '':
        return None
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, date):
        return val
    val_str = str(val).strip()
    for fmt in ['%d.%m.%Y', '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%dT%H:%M:%S']:
        try:
            return datetime.strptime(val_str, fmt).date()
        except ValueError:
            continue
    return None

def parse_date_range(val) -> tuple[date | None, date | None]:
    if pd.isna(val) or str(val).strip() == '':
        return None, None
    val_str = str(val).strip()
    if '-' in val_str and '.' in val_str:
        parts = val_str.split('-')
        if len(parts) == 2:
            return clean_date(parts[0].strip()), clean_date(parts[1].strip())
    return clean_date(val_str), None

def clean_fio(val) -> str:
    if pd.isna(val) or str(val).strip() == '':
        return ''
    val_str = str(val).strip()
    match = re.match(r'^[A-Za-zА-Яа-яЁё\s\-]+', val_str)
    if match:
        return match.group(0).strip()
    return val_str

def normalize_fio(fio: str) -> str:
    if not fio:
        return ''
    return re.sub(r'\s+', ' ', fio.lower().strip()).title()

def clean_tech_name(tech_name):
    """
    Нормализует название технологии (как в Парсинг_v4):
    - Приводит к верхнему регистру
    - Заменяет кириллические символы на латинские через str.translate()
    - Удаляет спецсимволы и пробелы
    - Объединяет синонимы (QAJAVA -> AQAJAVA)
    """
    if pd.isna(tech_name):
        return tech_name
    tech_str = str(tech_name).upper()
    cyrillic_to_latin = str.maketrans({
        'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M',
        'Н': 'H', 'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T',
        'Х': 'X',
    })
    tech_str = tech_str.translate(cyrillic_to_latin)
    tech_str = re.sub(r'[^A-Z0-9]', '', tech_str)
    tech_str = re.sub(r'^Q?AQ?AJAVA$', 'AQAJAVA', tech_str)
    return tech_str

def is_valid_techlead(name: str) -> bool:
    """
    Проверяет, что имя техлида выглядит как реальное 'Имя Фамилия',
    а не артефакт таблицы ( '-', 'х', '2-я технология' и т.д.)
    """
    if pd.isna(name):
        return False
    name = str(name).strip()
    if len(name) < 3:
        return False
    if " " not in name:
        return False
    blacklist = ["-", "х", "x", "2-я технология", "вторая технология", "команда", "techleads"]
    if name.lower() in blacklist:
        return False
    return True

def extract_sheet_data(worksheet) -> pd.DataFrame:
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()

    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > COL_NAME:
            col_b = str(row[COL_TECH]).strip().lower()
            col_c = str(row[COL_TECHLEAD]).strip().lower()
            col_d = str(row[COL_NAME]).strip().lower()
            if ('технология' in col_b and
                ('техлид' in col_c or 'techlead' in col_c) and
                ('фамилия' in col_d or 'фио' in col_d)):
                header_idx = i
                break
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= COL_REASON:
            row.append('')
        col_b_val = str(row[COL_TECH]).strip()
        if not col_b_val:
            continue
        fio = clean_fio(row[COL_NAME]).strip()
        if not fio:
            continue
        data.append({
            'Технология': col_b_val,
            'Техлид': clean_fio(row[COL_TECHLEAD]),
            'Лаборант': fio,
            'Дата старта': clean_date(row[COL_START]),
            'Дата стопа': clean_date(row[COL_STOP]),
            'Причина_статус': str(row[COL_REASON]).strip().lower(),
            'Источник': worksheet.title,
        })
    return pd.DataFrame(data)

def extract_interim_status_data(worksheet) -> pd.DataFrame:
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()

    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > INTERIM_COL_NAME:
            col_c = str(row[INTERIM_COL_TECH]).strip().lower()
            col_d = str(row[INTERIM_COL_TECHLEAD]).strip().lower()
            col_e = str(row[INTERIM_COL_NAME]).strip().lower()
            if ('технология' in col_c and
                ('техлид' in col_d or 'techlead' in col_d) and
                ('фамилия' in col_e or 'фио' in col_e)):
                header_idx = i
                break
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= INTERIM_COL_REASON:
            row.append('')
        col_c_val = str(row[INTERIM_COL_TECH]).strip()
        if not col_c_val:
            continue
        fio = clean_fio(row[INTERIM_COL_NAME]).strip()
        if not fio:
            continue
        reason = str(row[INTERIM_COL_REASON]).strip().lower()
        if not reason:
            continue
        start_date, end_date = parse_date_range(row[INTERIM_COL_STOP])
        data.append({
            'Технология': col_c_val,
            'Техлид': clean_fio(row[INTERIM_COL_TECHLEAD]),
            'Лаборант': fio,
            'Дата старта': start_date if start_date else clean_date(row[INTERIM_COL_START]),
            'Дата стопа': end_date if end_date else clean_date(row[INTERIM_COL_STOP]),
            'Причина_статус': reason,
            'Источник': worksheet.title,
        })
    return pd.DataFrame(data)

In [3]:
# Ячейка 3: Формирование снепшота, история и журнал изменений

def prepare_snapshot_df(df, snapshot_date, kind) -> pd.DataFrame:
    """
    Приводит спарсенные данные к единому формату снепшота.
    НОВОЕ: применяет валидацию техлидов (как в Парсинг_v4).
    """
    # Фильтрация невалидных техлидов
    df = df[df['Техлид'].apply(is_valid_techlead)].copy()

    out = pd.DataFrame({
        "Дата_снепшота": snapshot_date.isoformat(),
        "Вид_записи": kind,
        "Источник": df["Источник"].values,
        "Технология_норм": df["Технология"].apply(clean_tech_name).values,
        "Технология_оригинал": df["Технология"].str.strip().values,
        "Техлид_норм": df["Техлид"].apply(clean_fio).values,
        "Техлид_оригинал": df["Техлид"].str.strip().values,
        "Лаборант_норм": df["Лаборант"].apply(normalize_fio).values,
        "Лаборант_оригинал": df["Лаборант"].str.strip().values,
        "Дата_старта": df["Дата старта"].apply(lambda d: d.isoformat() if d else "").values,
        "Дата_стопа": df["Дата стопа"].apply(lambda d: d.isoformat() if d else "").values,
        "Причина_статус": df["Причина_статус"].fillna("").str.strip().values,
    })
    return out[SNAPSHOT_COLUMNS]

def save_daily_file(snap_df, snapshot_date) -> Path:
    path = SNAPSHOT_DIR / f"snap_{snapshot_date.isoformat()}.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        snap_df.to_excel(writer, sheet_name="Снепшот", index=False)
        for i, col in enumerate(writer.sheets["Снепшот"].columns):
            max_len = max(len(str(c.value)) for c in col if c.value is not None)
            writer.sheets["Снепшот"].column_dimensions[chr(65 + i)].width = min(max_len + 2, 35)
    return path

def load_history() -> pd.DataFrame:
    if HISTORY_FILE.exists():
        return pd.read_csv(HISTORY_FILE, dtype=str, keep_default_na=False)
    return pd.DataFrame(columns=SNAPSHOT_COLUMNS)

def append_to_history(snap_df, snapshot_date) -> int:
    hist = load_history()
    hist = hist[hist["Дата_снепшота"] != snapshot_date.isoformat()]
    combined = pd.concat([hist, snap_df], ignore_index=True)
    combined.to_csv(HISTORY_FILE, index=False, encoding="utf-8-sig")
    return len(combined)

def get_previous_snapshot(snapshot_date):
    hist = load_history()
    hist = hist[hist["Дата_снепшота"] < snapshot_date.isoformat()]
    if hist.empty:
        return None, None
    prev_date = hist["Дата_снепшота"].max()
    return prev_date, hist[hist["Дата_снепшота"] == prev_date]

def compute_changes(prev_df, curr_df) -> pd.DataFrame:
    prev = prev_df[KEY_COLS + COMPARE_COLS].drop_duplicates(KEY_COLS)
    curr = curr_df[KEY_COLS + COMPARE_COLS].drop_duplicates(KEY_COLS)
    m = prev.merge(curr, on=KEY_COLS, how="outer",
                   suffixes=("_prev", "_curr"), indicator=True)
    rows = []
    for _, r in m.iterrows():
        key = {c: r[c] for c in KEY_COLS}
        old = " | ".join(str(r[f"{c}_prev"]) for c in COMPARE_COLS)
        new = " | ".join(str(r[f"{c}_curr"]) for c in COMPARE_COLS)
        if r["_merge"] == "right_only":
            rows.append({**key, "Тип_изменения": "добавлен", "Было": "", "Стало": new})
        elif r["_merge"] == "left_only":
            rows.append({**key, "Тип_изменения": "удален", "Было": old, "Стало": ""})
        else:
            if any(r[f"{c}_prev"] != r[f"{c}_curr"] for c in COMPARE_COLS):
                rows.append({**key, "Тип_изменения": "изменен", "Было": old, "Стало": new})
    return pd.DataFrame(rows, columns=KEY_COLS + ["Тип_изменения", "Было", "Стало"]) \
        if rows else pd.DataFrame(columns=KEY_COLS + ["Тип_изменения", "Было", "Стало"])

def append_changes(changes_df, snapshot_date):
    if changes_df.empty:
        return
    changes_df = changes_df.copy()
    changes_df.insert(0, "Дата_снепшота", snapshot_date.isoformat())
    changes_df = changes_df[CHANGES_COLUMNS]
    if CHANGES_FILE.exists():
        log = pd.read_csv(CHANGES_FILE, dtype=str, keep_default_na=False)
        log = log[log["Дата_снепшота"] != snapshot_date.isoformat()]
        changes_df = pd.concat([log, changes_df], ignore_index=True)
    changes_df.to_csv(CHANGES_FILE, index=False, encoding="utf-8-sig")

In [4]:
# Ячейка 4: Главная функция

def main(snapshot_date: date | None = None):
    snapshot_date = snapshot_date or date.today()
    print(f"Формирование снепшота за {snapshot_date.isoformat()}...")

    try:
        client = authenticate_with_service_account('credentials.json')
        spreadsheet = client.open_by_key(SPREADSHEET_ID)
        print(f"Таблица открыта: {spreadsheet.title}")
    except Exception as e:
        print(f"ОШИБКА подключения: {e}")
        return

    parts = []
    for sheet_name in SHEET_NAMES:
        try:
            df = extract_sheet_data(spreadsheet.worksheet(sheet_name))
            if not df.empty:
                parts.append(prepare_snapshot_df(df, snapshot_date, "проект"))
                print(f"  Лист '{sheet_name}': {len(df)} строк (до фильтрации техлидов)")
        except Exception as e:
            print(f"  Ошибка на листе {sheet_name}: {e}")

    try:
        interim_df = extract_interim_status_data(spreadsheet.worksheet(INTERIM_STATUS_SHEET))
        if not interim_df.empty:
            parts.append(prepare_snapshot_df(interim_df, snapshot_date, "промежуточный статус"))
            print(f"  Лист '{INTERIM_STATUS_SHEET}': {len(interim_df)} записей")
    except Exception as e:
        print(f"  Ошибка на листе {INTERIM_STATUS_SHEET}: {e}")

    if not parts:
        print("ОШИБКА: нет данных для снепшота.")
        return

    snap_df = pd.concat(parts, ignore_index=True)
    print(f"После фильтрации невалидных техлидов в снепшоте: {len(snap_df)} строк")

    # Журнал изменений относительно предыдущего дня
    prev_date, prev_df = get_previous_snapshot(snapshot_date)
    if prev_df is not None:
        changes = compute_changes(prev_df, snap_df)
        print(f"\nСравнение с предыдущим снепшотом от {prev_date}:")
        if changes.empty:
            print("  изменений нет")
        else:
            for t in ["добавлен", "удален", "изменен"]:
                n = (changes["Тип_изменения"] == t).sum()
                if n:
                    print(f"  {t}: {n}")
            append_changes(changes, snapshot_date)
    else:
        print("\nПредыдущий снепшот не найден — журнал изменений не формируется")

    # Сохранение: дневной файл + сквозная история
    path = save_daily_file(snap_df, snapshot_date)
    total = append_to_history(snap_df, snapshot_date)

    print(f"\nСнепшот сохранен: {path} ({len(snap_df)} строк)")
    print(f"Всего записей в истории: {total}")

if __name__ == "__main__":
    main()
    # Для бэкфилла истории за прошлый день:
    # main(date(2026, 10, 13))

Формирование снепшота за 2026-10-14...
Таблица открыта: Копия ФИДБЕКИ Aston
  Лист 'DEMO Лист': 16 строк (до фильтрации техлидов)
  Лист 'A-Benefits': 16 строк (до фильтрации техлидов)
  Лист '1C': 43 строк (до фильтрации техлидов)
  Лист 'Active Bank': 51 строк (до фильтрации техлидов)
  Лист 'A-Sber': 30 строк (до фильтрации техлидов)
  Лист 'Betta Bank': 30 строк (до фильтрации техлидов)
  Лист 'Evergreen Bank+Laba ABS': 52 строк (до фильтрации техлидов)
  Лист 'Liberty Bank': 57 строк (до фильтрации техлидов)
  Лист 'Investment': 46 строк (до фильтрации техлидов)
  Лист 'L&D': 11 строк (до фильтрации техлидов)
  Лист 'MyCare': 27 строк (до фильтрации техлидов)
  Лист 'SQL Trainee': 22 строк (до фильтрации техлидов)
  Лист 'ProBusiness': 21 строк (до фильтрации техлидов)
  Лист 'Промежуточный статус стажера': 142 записей
После фильтрации невалидных техлидов в снепшоте: 463 строк

Предыдущий снепшот не найден — журнал изменений не формируется

Снепшот сохранен: snapshots\snap_2026-10